In [ ]:
import faster_whisper
from faster_whisper import WhisperModel

model = WhisperModel("../../audio_models/whisper/whisper-tiny/", device="cpu", compute_type="int8")
audio_file_path = "../../datasets/audio_dataset/test_1.mp3"

result = model.transcribe(audio_file_path)
print(result["text"])

In [3]:
from transformers import WhisperProcessor
from transformers import WhisperForConditionalGeneration
from datasets import load_dataset

# load model and processor
whisper_model_path = "../../audio_models/whisper/whisper-tiny/"
processor = WhisperProcessor.from_pretrained(whisper_model_path)
model = WhisperForConditionalGeneration.from_pretrained(whisper_model_path)
model.config.forced_decoder_ids = None

In [5]:
# load dummy dataset and read audio files
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
sample = ds[0]["audio"]
input_features = processor(sample["array"], sampling_rate=sample["sampling_rate"], return_tensors="pt").input_features

In [6]:
# generate token ids
predicted_ids = model.generate(input_features)
# decode token ids to text
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=False)
print(transcription)

Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[' Mr. Quilter is the apostle of the middle classes and we are glad to welcome his gospel.']


In [7]:
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
print(transcription)

[' Mr. Quilter is the apostle of the middle classes and we are glad to welcome his gospel.']


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
import torchaudio  # để đọc file .mp3
import os

whisper_model_path = "../../audio_models/whisper/whisper-tiny/"
processor = WhisperProcessor.from_pretrained(whisper_model_path)
model = WhisperForConditionalGeneration.from_pretrained(whisper_model_path)
model.config.forced_decoder_ids = None  # Cho phép tự phát hiện ngôn ngữ

audio_path = "../../datasets/audio_dataset/test_1.mp3"
waveform, sample_rate = torchaudio.load(audio_path)

if sample_rate != 16000:
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
    waveform = resampler(waveform)
    sample_rate = 16000

input_features = processor(waveform.squeeze().numpy(), sampling_rate=sample_rate, return_tensors="pt").input_features

predicted_ids = model.generate(input_features)

transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print("🔊 Transcription:", transcription)


🔊 Transcription:  Hey everybody, welcome to this B2 English Listening Practice video. You can use this video to train your listening and comprehension as I speak. You ready? So today I'm going to talk to you about living abroad. I was an expert for a few years, so I can offer a pretty good amount of insight into this topic. Let's start first with the good things. Living abroad opens up a whole world of new experiences, ideas and cultures that you never would have dreamed of if you hadn't moved abroad. The second you step off the plane and realize that you're not just on
